## Simplified perception pipeline for MP4

In [1]:
import time
start_time = time.perf_counter()

In [2]:
import sys
import os
sys.path.append('../../')
import json
import time
import math

import numpy as np
import ctypes

import gi
gi.require_version('Gst', '1.0')
from gi.repository import GObject, GLib, Gst
from common.platform_info import PlatformInfo
from common.bus_call import bus_call
import pyds, random

In [3]:
from common_utils import *

Model Inference configuration

In [4]:
# pgie_config_file = "../../models/configs/config_infer_primary_yolo26.txt"
pgie_config_file = "../../models/configs/config_infer_primary_rfdetr.txt"
tracker_config_file = "../../models/configs/tracker.txt"

In [5]:
from common_utils import *

INVALID_TRACK_ID = 18446744073709551615
PGIE_GIE_ID = 1
VEHICLE_CLASS_IDS = {2, 3, 5, 7}

In [6]:
KAFKA_OUTPUT_JSONL = "kafka_output.jsonl"

from pathlib import Path
Path(KAFKA_OUTPUT_JSONL).unlink(missing_ok=True)


In [7]:
def bus_call(bus, message, loop):
    t = message.type
    if t == Gst.MessageType.EOS:
        sys.stdout.write("End-of-stream")
        loop.quit()
    elif t==Gst.MessageType.WARNING:
        err, debug = message.parse_warning()
        sys.stderr.write("Warning: %s: %s\n" % (err, debug))
    elif t == Gst.MessageType.ERROR:
        err, debug = message.parse_error()
        sys.stderr.write("Error: %s: %s\n" % (err, debug))
        loop.quit()
    return True

In [8]:
platform_info = PlatformInfo()
Gst.init(None)

print("Creating Pipeline \n ")
pipeline = Gst.Pipeline()

if not pipeline:
    sys.stderr.write(" Unable to create Pipeline \n")

Creating Pipeline 
 


In [9]:
source=Gst.ElementFactory.make("filesrc", "file-source")
source.set_property('location', INPUT_FILE)

h264parser=Gst.ElementFactory.make("h264parse", "h264-parser")

decoder=Gst.ElementFactory.make("nvv4l2decoder", "nvv4l2-decoder")

streammux=Gst.ElementFactory.make("nvstreammux", "stream-muxer")
streammux.set_property('width', BBOX_COORD_WIDTH)
streammux.set_property('height', BBOX_COORD_HEIGHT)

streammux.set_property('batched-push-timeout', 4000000)
streammux.set_property('batch-size', 1)

fakesink = Gst.ElementFactory.make("fakesink", "metadata-fakesink")
if not fakesink:
    raise RuntimeError("Unable to create fakesink")

fakesink.set_property("sync", False)
fakesink.set_property("async", False)


print('Created elements')

Created elements


#### Object Detection - PGIE

In [10]:
pgie=Gst.ElementFactory.make("nvinfer", "primary-inference")
pgie.set_property('config-file-path', pgie_config_file)

# We need to rebuild the engine file when we change the batch size
# pgie.set_property("batch-size", num_sources)

In [11]:
import configparser

tracker = Gst.ElementFactory.make("nvtracker", "tracker")

config = configparser.ConfigParser()
config.read(tracker_config_file)
config.sections()

for key in config['tracker']:
    if key == 'tracker-width' :
        tracker_width = config.getint('tracker', key)
        tracker.set_property('tracker-width', tracker_width)
    if key == 'tracker-height' :
        tracker_height = config.getint('tracker', key)
        tracker.set_property('tracker-height', tracker_height)
    if key == 'gpu-id' :
        tracker_gpu_id = config.getint('tracker', key)
        tracker.set_property('gpu-id', tracker_gpu_id)
    if key == 'll-lib-file' :
        tracker_ll_lib_file = config.get('tracker', key)
        tracker.set_property('ll-lib-file', tracker_ll_lib_file)
    if key == 'll-config-file' :
        tracker_ll_config_file = config.get('tracker', key)
        tracker.set_property('ll-config-file', tracker_ll_config_file)

#### Metadata probe function on tracker

In [12]:
import json
import time
from pathlib import Path


class JsonlMetadataWriter:
    def __init__(self, output_path: str, flush_every: int = 30):
        self.output_path = Path(output_path)
        self.output_path.parent.mkdir(parents=True, exist_ok=True)

        self.f = open(self.output_path, "a", buffering=1)
        self.count = 0
        self.flush_every = flush_every

    def write(self, record: dict):
        self.f.write(json.dumps(record, separators=(",", ":")) + "\n")
        self.count += 1

        if self.count % self.flush_every == 0:
            self.f.flush()

    def close(self):
        if not self.f.closed:
            self.f.flush()
            self.f.close()

In [13]:
def get_frame_timestamp(frame_meta):
    try:
        ntp_ns = int(frame_meta.ntp_timestamp)
        if ntp_ns > 0:
            return ntp_ns / 1e9
    except Exception:
        pass

    return None

def build_object_record(obj_meta, camera_id: str):
    object_id = int(obj_meta.object_id)

    obj = {
        "object_id": object_id,
        "track_id": f"{camera_id}_{object_id}" if object_id >= 0 else None,

        "unique_component_id": int(obj_meta.unique_component_id),
        "class_id": int(obj_meta.class_id),

        "det_conf": float(obj_meta.confidence),
        "tracker_confidence": float(obj_meta.tracker_confidence),
    }

    return obj

def metadata_export_probe(pad, info, user_data):
    writer = user_data["writer"]
    camera_id = user_data.get("camera_id", "cam_0")
    only_vehicles = user_data.get("only_vehicles", True)
    sample_every_n_frames = user_data.get("sample_every_n_frames", 1)

    gst_buffer = info.get_buffer()
    if not gst_buffer:
        return Gst.PadProbeReturn.OK

    batch_meta = pyds.gst_buffer_get_nvds_batch_meta(hash(gst_buffer))
    if not batch_meta:
        return Gst.PadProbeReturn.OK

    l_frame = batch_meta.frame_meta_list

    while l_frame is not None:
        try:
            frame_meta = pyds.NvDsFrameMeta.cast(l_frame.data)
        except StopIteration:
            break

        frame_idx = int(frame_meta.frame_num)

        if sample_every_n_frames > 1 and frame_idx % sample_every_n_frames != 0:
            try:
                l_frame = l_frame.next
                continue
            except StopIteration:
                break

        source_id = int(frame_meta.source_id)
        source_id_to_camera_id = user_data.get("source_id_to_camera_id", {})
        camera_id = source_id_to_camera_id.get(source_id, f"source_{source_id}")

        frame_record = {
            "camera_id": camera_id,
            "source_id": source_id,
            "frame_idx": frame_idx,
            "ntp_timestamp": get_frame_timestamp(frame_meta),
            "objects": [],
        }

        l_obj = frame_meta.obj_meta_list

        while l_obj is not None:
            try:
                obj_meta = pyds.NvDsObjectMeta.cast(l_obj.data)
            except StopIteration:
                break

            if int(obj_meta.unique_component_id) != PGIE_GIE_ID:
                try:
                    l_obj = l_obj.next
                    continue
                except StopIteration:
                    break

            class_id = int(obj_meta.class_id)

            if only_vehicles and class_id not in VEHICLE_CLASS_IDS:
                try:
                    l_obj = l_obj.next
                    continue
                except StopIteration:
                    break

            obj_record = build_object_record(obj_meta, camera_id)
            frame_record["objects"].append(obj_record)

            try:
                l_obj = l_obj.next
            except StopIteration:
                break

        # Write only frames with objects
        if frame_record["objects"]:
            writer.write(frame_record)
            print(f"writing frame : {frame_record["camera_id"]}_{frame_record["frame_idx"]}")

        try:
            l_frame = l_frame.next
        except StopIteration:
            break

    return Gst.PadProbeReturn.OK

In [14]:
metadata_writer = JsonlMetadataWriter(
    output_path=KAFKA_OUTPUT_JSONL,
    flush_every=30,
)

probe_data = {
    "writer": metadata_writer,
    "only_vehicles": True,
    "sample_every_n_frames": 1,
}


tracker_src_pad = tracker.get_static_pad("src")
if not tracker_src_pad:
    raise RuntimeError("Unable to get tracker src pad")

tracker_src_pad.add_probe(
    Gst.PadProbeType.BUFFER,
    metadata_export_probe,
    probe_data,
)

1

In [15]:
pipeline.add(source)
pipeline.add(h264parser)
pipeline.add(decoder)
pipeline.add(streammux)

pipeline.add(pgie)
pipeline.add(tracker)

pipeline.add(fakesink)
print('Added elements to pipeline')

Added elements to pipeline


In [16]:
def must_link(src, dst, name):
    ok = src.link(dst)
    print(f"[LINK] {name}: {ok}")
    if not ok:
        raise RuntimeError(f"Failed to link: {name}")


must_link(source, h264parser, "source -> h264parser")
must_link(h264parser, decoder, "h264parser -> decoder")

decoder_srcpad = decoder.get_static_pad("src")
streammux_sinkpad = streammux.get_request_pad("sink_0")

ret = decoder_srcpad.link(streammux_sinkpad)
print(f"[PAD LINK] decoder -> streammux: {ret}")
if ret != Gst.PadLinkReturn.OK:
    raise RuntimeError(f"Failed to link decoder -> streammux: {ret}")

must_link(streammux, pgie, "streammux -> pgie")
must_link(pgie, tracker, "pgie -> tracker")
must_link(tracker, fakesink, "tracker -> fakesink")

print("Linked elements in pipeline")

[LINK] source -> h264parser: True
[LINK] h264parser -> decoder: True
[PAD LINK] decoder -> streammux: <enum GST_PAD_LINK_OK of type Gst.PadLinkReturn>
[LINK] streammux -> pgie: True
[LINK] pgie -> tracker: True
[LINK] tracker -> fakesink: True
Linked elements in pipeline


/tmp/ipykernel_477/1340074502.py:12: DeprecationWarning: Gst.Element.get_request_pad is deprecated
  streammux_sinkpad = streammux.get_request_pad("sink_0")


In [17]:
loop=GLib.MainLoop()
bus=pipeline.get_bus()
bus.add_signal_watch()
bus.connect("message", bus_call, loop)
print('Added bus message handler')

Added bus message handler


### Run pipeline

In [18]:
# Start play back and listen to events
print("Starting pipeline")
pipeline.set_state(Gst.State.PLAYING)
try:
    loop.run()
except:
    pass

# Cleaning up as the pipeline comes to an end
pipeline.set_state(Gst.State.NULL)

Starting pipeline
Opening in BLOCKING MODE 
gstnvtracker: Loading low-level lib at /opt/nvidia/deepstream/deepstream/lib/libnvds_nvmultiobjecttracker.so
[NvMultiObjectTracker] Initialized


0:00:00.342962550   477     0x2ddce580 WARN                 nvinfer gstnvinfer.cpp:682:gst_nvinfer_logger:<primary-inference> NvDsInferContext[UID 1]: Warning from NvDsInferContextImpl::deserializeEngineAndBackend() <nvdsinfer_context_impl.cpp:2097> [UID = 1]: deserialize engine from file :/opt/nvidia/deepstream/deepstream-8.0/samples/playground/models/configs/../weights/rfdetr_large_2026.onnx_b1_gpu0_fp32.engine failed
0:00:00.342978450   477     0x2ddce580 WARN                 nvinfer gstnvinfer.cpp:682:gst_nvinfer_logger:<primary-inference> NvDsInferContext[UID 1]: Warning from NvDsInferContextImpl::generateBackendContext() <nvdsinfer_context_impl.cpp:2202> [UID = 1]: deserialize backend context from engine from file :/opt/nvidia/deepstream/deepstream-8.0/samples/playground/models/configs/../weights/rfdetr_large_2026.onnx_b1_gpu0_fp32.engine failed, try rebuild
0:00:00.342981703   477     0x2ddce580 INFO                 nvinfer gstnvinfer.cpp:685:gst_nvinfer_logger:<primary-inferenc


Building the TensorRT Engine

Building complete



0:00:16.947482846   477     0x2ddce580 INFO                 nvinfer gstnvinfer.cpp:685:gst_nvinfer_logger:<primary-inference> NvDsInferContext[UID 1]: Info from NvDsInferContextImpl::buildModel() <nvdsinfer_context_impl.cpp:2155> [UID = 1]: serialize cuda engine to file: /opt/nvidia/deepstream/deepstream-8.0/samples/playground/notebooks/topk/model_b1_gpu0_fp32.engine successfully
0:00:17.375308162   477     0x2ddce580 INFO                 nvinfer gstnvinfer_impl.cpp:343:notifyLoadModelStatus:<primary-inference> [UID 1]: Load new model:../../models/configs/config_infer_primary_rfdetr.txt sucessfully


writing frame : source_0_2
writing frame : source_0_3
writing frame : source_0_4
writing frame : source_0_5
writing frame : source_0_6
writing frame : source_0_7
writing frame : source_0_8
writing frame : source_0_9
writing frame : source_0_10
writing frame : source_0_11
writing frame : source_0_12
writing frame : source_0_13
writing frame : source_0_14
writing frame : source_0_15
writing frame : source_0_16
writing frame : source_0_17
writing frame : source_0_18
writing frame : source_0_19
writing frame : source_0_20
writing frame : source_0_21
writing frame : source_0_22
writing frame : source_0_23
writing frame : source_0_24
writing frame : source_0_25
writing frame : source_0_26
writing frame : source_0_27
writing frame : source_0_28
writing frame : source_0_29
writing frame : source_0_30
writing frame : source_0_31
writing frame : source_0_32
writing frame : source_0_33
writing frame : source_0_34
writing frame : source_0_35
writing frame : source_0_36
writing frame : source_0_37


<enum GST_STATE_CHANGE_SUCCESS of type Gst.StateChangeReturn>

In [19]:
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"Process finished in {elapsed_time:.4f} seconds")

Process finished in 41.9621 seconds


In [20]:
# !pip install onnx

In [21]:
# import onnx

# model_path = "/opt/nvidia/deepstream/deepstream-8.0/samples/playground/models/weights/rfdetr_large_compressed.onnx"

# # 1. Load the model
# model = onnx.load(model_path)

# # 2. Check for structural errors
# try:
#     onnx.checker.check_model(model)
#     print("ONNX structure is valid!")
# except onnx.checker.ValidationError as e:
#     print(f"ONNX structure is INVALID: {e}")

# # 3. Print inputs and outputs
# print("\n--- Model Inputs ---")
# for inp in model.graph.input:
#     print(f"Name: {inp.name}")
# print("\n--- Model Outputs ---")
# for out in model.graph.output:
#     print(f"Name: {out.name}")
